In [2]:
import pandas as pd
import itertools
import statsmodels.api as sm
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [12]:
def exhaustive_regression(X, y, categorical_target=False):
    """
    Uses:
      - OLS for continuous targets
      - MNLogit for multiclass categorical targets

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target vector

    Returns
    -------
    results : pd.DataFrame
        DataFrame with feature1, feature2, model type,
        interaction coefficients, and p-values.
    """
    feature_names = X.columns.tolist()

    # Detect target type
    if categorical_target:
        model_type = "multiclass"
        le = LabelEncoder()
        y = le.fit_transform(y)
        classes = le.classes_
    else:
        if pd.api.types.is_numeric_dtype(y):
            model_type = "continuous"
        else:
            model_type = "multiclass"
            le = LabelEncoder()
            y = le.fit_transform(y)
            classes = le.classes_

    results = []

    for f1, f2 in list(itertools.combinations(range(X.shape[1]), 2)):
        # Build design matrix
        feature_pair = X[[feature_names[f1], feature_names[f2]]].copy()
        feature_pair["interaction"] = X[feature_names[f1]] * X[feature_names[f2]]
        feature_pair = sm.add_constant(feature_pair)

        try: 

            if model_type == "continuous":
                # Ordinary Least Squares
                model = sm.OLS(y, feature_pair).fit()
                results.append({
                    "feature1": feature_names[f1],
                    "feature2": feature_names[f2],
                    "model": "OLS",
                    "interaction_coef": model.params.get("interaction", np.nan),
                    "interaction_pval": model.pvalues.get("interaction", np.nan)
                })
            else:
                # Multinomial Logistic Regression
                model = sm.MNLogit(y, feature_pair).fit(disp=False)
                if "interaction" in model.pvalues.index:
                    for class_idx, p in model.pvalues.loc["interaction"].items():
                        results.append({
                            "feature1": feature_names[f1],
                            "feature2": feature_names[f2],
                            "model": "MNLogit",
                            "class": classes[class_idx],
                            "interaction_coef": model.params.loc["interaction", class_idx],
                            "interaction_pval": model.pvalues.loc["interaction", class_idx]
                        })

        except Exception as e:
            results.append({
                "feature1": feature_names[f1],
                "feature2": feature_names[f2],
                "model": model_type,
                "error": str(e)
            })

    return pd.DataFrame(results)

In [22]:
def exhaustive_regression(X, y, categorical_target=False, alpha=0.1):
    """
    Uses:
      - OLS for continuous targets
      - MNLogit for multiclass categorical targets

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target vector
    categorical_target : bool
        If True, force treating y as categorical.
    alpha : float
        Significance level for flagging after Bonferroni correction.

    Returns
    -------
    results_df : pd.DataFrame
        DataFrame with feature1, feature2, model type,
        interaction coefficients, raw p-values, Bonferroni-adjusted p-values,
        and boolean indicator of significance after correction.
    """
    feature_names = X.columns.tolist()

    # Detect target type
    if categorical_target:
        model_type = "multiclass"
        le = LabelEncoder()
        y_enc = le.fit_transform(y)
        classes = le.classes_
    else:
        if pd.api.types.is_numeric_dtype(y):
            model_type = "continuous"
            y_enc = y.values
        else:
            model_type = "multiclass"
            le = LabelEncoder()
            y_enc = le.fit_transform(y)
            classes = le.classes_

    results = []

    for f1, f2 in itertools.combinations(range(X.shape[1]), 2):
        # Build design matrix
        fname1 = feature_names[f1]
        fname2 = feature_names[f2]
        feature_pair = X[[fname1, fname2]].copy()
        feature_pair["interaction"] = X[fname1] * X[fname2]
        feature_pair = sm.add_constant(feature_pair)

        try:
            if model_type == "continuous":
                # Ordinary Least Squares
                model = sm.OLS(y_enc, feature_pair).fit()
                pval = model.pvalues.get("interaction", np.nan)
                coef = model.params.get("interaction", np.nan)
                results.append({
                    "feature1": fname1,
                    "feature2": fname2,
                    "model": "OLS",
                    "class": None,
                    "interaction_coef": coef,
                    "interaction_pval": pval
                })
            else:
                # Multinomial Logistic Regression
                model = sm.MNLogit(y_enc, feature_pair).fit(disp=False)
                # model.params and model.pvalues are DataFrames indexed by parameter name,
                # columns correspond to classes (0..K-1)
                if "interaction" in model.pvalues.index:
                    # iterate columns (class indices)
                    for class_idx in model.pvalues.columns:
                        pval = model.pvalues.loc["interaction", class_idx]
                        coef = model.params.loc["interaction", class_idx]
                        # map class_idx (int) to actual class label if available
                        class_label = classes[class_idx] if 'classes' in locals() else class_idx
                        results.append({
                            "feature1": fname1,
                            "feature2": fname2,
                            "model": "MNLogit",
                            "class": class_label,
                            "interaction_coef": coef,
                            "interaction_pval": pval
                        })
                else:
                    # No interaction parameter found in results (unexpected), append NaNs
                    results.append({
                        "feature1": fname1,
                        "feature2": fname2,
                        "model": "MNLogit",
                        "class": None,
                        "interaction_coef": np.nan,
                        "interaction_pval": np.nan
                    })

        except Exception as e:
            # keep record of the failure — interaction_pval left as NaN
            results.append({
                "feature1": fname1,
                "feature2": fname2,
                "model": model_type,
                "class": None,
                "interaction_coef": np.nan,
                "interaction_pval": np.nan,
                "error": str(e)
            })

    results_df = pd.DataFrame(results)

    # Count number of tests performed (non-NaN p-values)
    m = results_df["interaction_pval"].notna().sum()

    # If no tests completed, return early
    if m == 0:
        results_df["interaction_pval_bonf"] = np.nan
        results_df["bonf_significant"] = False
        return results_df

    # Bonferroni adjustment: p_adjusted = min(1, p * m)
    results_df["interaction_pval_bonf"] = results_df["interaction_pval"].apply(
        lambda p: min(1.0, p * m) if pd.notna(p) else np.nan
    )

    # Flag significance after Bonferroni correction
    results_df["bonf_significant"] = results_df["interaction_pval_bonf"].apply(
        lambda p: bool(p < alpha) if pd.notna(p) else False
    )

    # Optional: also include raw alpha threshold flag for convenience
    results_df["raw_significant"] = results_df["interaction_pval"].apply(
        lambda p: bool(p < alpha) if pd.notna(p) else False
    )

    # Add metadata about number of tests and alpha used
    results_df.attrs["n_tests"] = int(m)
    results_df.attrs["alpha"] = float(alpha)

    return results_df

TEST (no perturbation)

In [9]:
pd.set_option('display.max_colwidth', None)

In [23]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Data/cell_cycle_tidied.csv")

# Define features and target
X = df.drop(columns=['phase', 'age', 'PHATE_1', 'PHATE_2'])  # Features
y = df['age']  # Target: age

In [46]:
# first 50 rows
X_small = X.iloc[:50, :]
y_small = y.iloc[:50]

# first 10 features
feature_subset = X.columns[:10] 
X_small = X[feature_subset].iloc[:50, :]
y_small = y.iloc[:50]

In [24]:
df = exhaustive_regression(X, y)

In [34]:
top20 = df.sort_values('interaction_pval').head(20)

KeyError: 'interaction_pval'

In [ ]:
top20

,feature1,feature2,model,interaction_coef,interaction_pval
4302,pRB..nuc.median.,p27..nuc.median.,OLS,-1.765153,0.000000e+00
6204,pp21..nuc.median.,p21..phospho.total.nuc.,OLS,-0.306167,8.255576e-292
5007,p27..nuc.median.,RB..phospho.total.nuc.,OLS,-1.552622,5.699296e-290
5029,p27..nuc.median.,ratio,OLS,-1.552622,5.699296e-290
6005,pp21..nuc.median.,pp65..nuc.median.,OLS,-1.068402,3.311855e-277
3320,RB..nuc.median.,p27..nuc.median.,OLS,-1.494585,7.761544e-263
6180,pp21..nuc.median.,ERK..phospho.total.cell.,OLS,1.852032,4.006552e-250
4795,p27..nuc.median.,DNA..nuc.median.,OLS,-1.535953,2.528862e-240
6224,pp21..nuc.median.,ratio,OLS,-2.926138,1.254877e-222
6202,pp21..nuc.median.,RB..phospho.total.nuc.,OLS,-2.926138,1.254877e-222


In [21]:
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/LOCO/cancer_iloco_xgb.csv")
significant_count = (df["scores"] - df["ci"] > 0).sum()
print(f"Number of significant interactions (p < 0.1): {significant_count}")

Number of significant interactions (p < 0.1): 82


In [13]:
top_20 = df.nlargest(20, "scores")
top_20

,feature,scores,ci
182,Intensity_MedianIntensity_cycD1 & Intensity_MedianIntensity_pRB,0.007844,0.000873
187,Intensity_MedianIntensity_p21 & Intensity_MedianIntensity_pRB,0.003412,0.001083
134,Intensity_MedianIntensity_ER & pRB_over_RB,0.003102,0.000634
87,Intensity_MedianIntensity_Cdh1 & Intensity_MedianIntensity_ER,0.002856,0.001709
157,Intensity_MedianIntensity_RB & Intensity_MedianIntensity_cycD1,0.002691,0.000393
138,Intensity_MedianIntensity_Ki67 & Intensity_MedianIntensity_cycA,0.002676,0.000964
63,Intensity_MedianIntensity_CDK4 & Intensity_MedianIntensity_cycA,0.002375,0.000621
36,Intensity_IntegratedIntensity_DNA & pRB_over_RB,0.002334,0.000216
69,Intensity_MedianIntensity_CDK4 & pRB_over_RB,0.002140,0.000621
181,Intensity_MedianIntensity_cycD1 & Intensity_MedianIntensity_p21,0.002129,0.000873


TEST (cancer)

In [26]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

# Separate features and target
X = df.drop(columns=['Metadata_well'])
X = X.select_dtypes(include=['number'])
y = df['Metadata_well']

In [27]:
df = exhaustive_regression(X, y, categorical_target=True)

In [30]:
n_sig = df["bonf_significant"].sum()
print(f"Number of Bonferroni-significant interactions: {n_sig}")

Number of Bonferroni-significant interactions: 395


In [37]:
top20 = df.sort_values('interaction_pval').head(20)

In [39]:
top20

,feature1,feature2,model,class,interaction_coef,interaction_pval
759,Intensity_MedianIntensity_pRB,pRB_over_RB,MNLogit,100,0.921525,0.000000e+00
671,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_pRB,MNLogit,100,0.912245,0.000000e+00
670,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_pRB,MNLogit,10,0.516427,0.000000e+00
695,Intensity_MedianIntensity_cycA,Intensity_MedianIntensity_pRB,MNLogit,100,0.785408,0.000000e+00
643,Intensity_MedianIntensity_RB,Intensity_MedianIntensity_pRB,MNLogit,100,1.124490,0.000000e+00
699,Intensity_MedianIntensity_cycA,pRB_over_RB,MNLogit,100,0.664718,0.000000e+00
395,Intensity_MedianIntensity_Cdh1,pRB_over_RB,MNLogit,100,0.853645,0.000000e+00
715,Intensity_MedianIntensity_cycB1,Intensity_MedianIntensity_pRB,MNLogit,100,0.564088,0.000000e+00
398,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_E2F1,MNLogit,10,0.433850,0.000000e+00
579,Intensity_MedianIntensity_Ki67,pRB_over_RB,MNLogit,100,1.191305,0.000000e+00


DOSE DEPENDANT

In [31]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

# Separate features and target
X = df.drop(columns=['phase'])
X = X.select_dtypes(include=['number'])
y = df['phase']

In [32]:
df = exhaustive_regression(X, y, categorical_target=True)

/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3027: RuntimeWarning: overflow encountered in exp
  eXB = np.column_stack((np.ones(len(X)), np.exp(X)))
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3028: RuntimeWarning: invalid value encountered in divide
  return eXB/eXB.sum(1)[:,None]
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3027: RuntimeWarning: overflow encountered in exp
  eXB = np.column_stack((np.ones(len(X)), np.exp(X)))
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3028: RuntimeWarning: invalid value encountered in divide
  return eXB/eXB.sum(1)[:,None]
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:3027: RuntimeWarning: overflow encountered in exp
  eXB = np.column

In [33]:
n_sig = df["bonf_significant"].sum()
print(f"Number of Bonferroni-significant interactions: {n_sig}")

Number of Bonferroni-significant interactions: 446


In [42]:
top20 = df.sort_values('interaction_pval').head(20)
top20

,feature1,feature2,model,class,interaction_coef,interaction_pval
462,Intensity_MedianIntensity_Ki67,Metadata_well,MNLogit,G0,0.007942,0.000000e+00
1,AreaShape_Area,Intensity_IntegratedIntensity_DNA,MNLogit,G1,-1.149329,0.000000e+00
2,AreaShape_Area,Intensity_IntegratedIntensity_DNA,MNLogit,G2/M,-0.967918,0.000000e+00
439,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_Skp2,MNLogit,G1,-0.473036,0.000000e+00
315,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_E2F1,MNLogit,G0,-0.407602,0.000000e+00
321,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Ki67,MNLogit,G0,-0.794763,0.000000e+00
322,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Ki67,MNLogit,G1,-1.073437,0.000000e+00
440,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_Skp2,MNLogit,G2/M,-0.681807,0.000000e+00
327,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_RB,MNLogit,G0,-0.404397,0.000000e+00
330,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Skp2,MNLogit,G0,-0.930711,0.000000e+00


In [20]:
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/LOCO/cancer_iloco_ensemble.csv")
significant_count = (df["scores"] - df["ci"] > 0).sum()
print(f"Number of significant interactions (p < 0.1): {significant_count}")

Number of significant interactions (p < 0.1): 82


In [15]:
top_20 = df.nlargest(20, "scores")
top_20

,feature,scores,ci
209,pRB_over_RB & Metadata_well,0.048161,0.002420
38,Intensity_IntegratedIntensity_DNA & Metadata_well,0.026614,0.002364
208,Intensity_MedianIntensity_pRB & Metadata_well,0.014119,0.000568
171,Intensity_MedianIntensity_RB & Intensity_MedianIntensity_pRB,0.010555,0.000386
173,Intensity_MedianIntensity_RB & Metadata_well,0.003133,0.000386
107,Intensity_MedianIntensity_Cdt1 & Intensity_MedianIntensity_Ki67,0.002143,0.000365
163,Intensity_MedianIntensity_PR & pRB_over_RB,0.002139,0.000217
144,Intensity_MedianIntensity_Ki67 & Intensity_MedianIntensity_PR,0.002053,0.000456
103,Intensity_MedianIntensity_Cdh1 & pRB_over_RB,0.001955,0.000385
193,Intensity_MedianIntensity_cycB1 & pRB_over_RB,0.001893,0.000545
